In [253]:
from pathlib import Path

import boto3
import pandas as pd
import numpy as np
from botocore.exceptions import BotoCoreError, ClientError

from secretkeys import ACCESS_KEY, KEY_ID

# Замените примеры значениями преподавателя.
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/train.csv"
LOCAL_PATH = Path("data/listing.csv")

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

In [254]:
def read_s3_csv(bucket, key):

    response=s3.get_object(Bucket=bucket, Key=key)
    df=pd.read_csv(response['Body'])
    return df

In [255]:
def read_data(bucket, key, local_path=None):
    try:
        df=read_s3_csv(bucket, key)
        return df

    except (BotoCoreError, ClientError, OSError) as e:
        if local_path is None:
            raise

        print(f"S3 error: {type(e).__name__}: {e}")
        print(f"Reading local file: {local_path}")

        df=pd.read_csv(local_path)
        return df

In [256]:
df_s3 = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)
display(df_s3.head())
print("Размер таблицы:", df_s3.shape)

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,1001248165793428398,2026-03-21,51.434110,-0.161020,2,1.0,1.0,1.0,1.0,Wandsworth,Entire serviced apartment,Entire home/apt,130.0
1,1001501494788436614,2026-03-21,51.514120,-0.176810,5,1.0,3.0,3.0,2.0,Westminster,Entire rental unit,Entire home/apt,232.0
2,1001555419418023705,2026-03-21,51.542042,0.121361,4,2.0,2.0,2.0,1.0,Barking and Dagenham,Entire condo,Entire home/apt,216.0
3,1001555573812356101,2026-03-21,51.540557,0.120994,2,1.0,1.0,1.0,1.0,Barking and Dagenham,Entire place,Entire home/apt,115.0
4,1001604905732387546,2026-03-21,51.377860,-0.086780,2,1.0,1.0,1.0,1.0,Croydon,Private room in home,Private room,86.0


Размер таблицы: (5724, 13)


In [257]:
df_s3.isna().sum()

listing_id                 0
snapshot_date              0
latitude                   0
longitude                  0
accommodates               0
bathrooms                  6
bedrooms                   7
beds                      21
minimum_nights             1
neighbourhood_cleansed     0
property_type              0
room_type                  0
price                      0
dtype: int64

In [258]:
df_s3['beds']=df_s3['beds'].fillna(np.round(df_s3['accommodates']/2))
df_s3['minimum_nights']=df_s3['minimum_nights'].fillna(df_s3.minimum_nights.min())

In [259]:
df_s3[df_s3['bedrooms'].isna()]

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
1270,1313113164113582510,2026-03-21,51.546039,-0.113163,2,1.0,NaN,2.0,1.0,Islington,Entire rental unit,Entire home/apt,113.70
1762,1399317296703966600,2026-03-21,51.529397,-0.080553,6,3.0,NaN,3.0,1.0,Hackney,Entire rental unit,Entire home/apt,408.00
2897,1576565282236600908,2026-03-21,51.597793,-0.076469,2,1.0,NaN,1.0,1.0,Haringey,Shared room in home,Shared room,21.68
3037,1597102814817923070,2026-03-21,51.531620,-0.285470,4,1.0,NaN,1.0,1.0,Brent,Shared room in rental unit,Shared room,19.57
3446,1637729315219860938,2026-03-21,51.531882,-0.285707,2,2.0,NaN,1.0,1.0,Brent,Shared room in rental unit,Shared room,36.00
3513,1644143554357367605,2026-03-21,51.498841,-0.163670,3,1.0,NaN,1.0,2.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,158.00
4467,52803341,2026-03-21,51.555380,-0.206020,4,1.0,NaN,3.0,6.0,Camden,Entire loft,Entire home/apt,124.96


In [260]:
def fill_bathrooms(df):
    df=df.copy()
    medians=df.groupby('bedrooms')['bathrooms'].transform('median').round()
    df['bathrooms']=df['bathrooms'].fillna(medians)
    return df

In [261]:
def fill_bedrooms(df):
    df=df.copy()
    medians=df.groupby('beds')['bedrooms'].transform('median').round()
    df['bedrooms']=df['bedrooms'].fillna(medians)
    return df
    

In [262]:
df=fill_bedrooms(fill_bathrooms(df_s3))
df.isna().sum()

listing_id                0
snapshot_date             0
latitude                  0
longitude                 0
accommodates              0
bathrooms                 0
bedrooms                  0
beds                      0
minimum_nights            0
neighbourhood_cleansed    0
property_type             0
room_type                 0
price                     0
dtype: int64

In [263]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

In [264]:
X=df.drop(['listing_id', 'snapshot_date', 'price'], axis=1)
y=df['price']

In [265]:
X_train, X_test, y_train, y_test=train_test_split(X, y, random_state=42, test_size=0.25, shuffle=True)

In [266]:
num_cols=X_train.select_dtypes(include=('number')).columns
cat_cols=X_train.select_dtypes(exclude=('number')).columns

In [267]:
scaler=StandardScaler()
scaler.set_output(transform="pandas")
X_train_scaled=scaler.fit_transform(X_train[num_cols])

In [268]:
X_test_scaled=scaler.transform(X_test[num_cols])

In [269]:
ohe=OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')
X_train_ohe=ohe.fit_transform(X_train[cat_cols])
col_names=ohe.get_feature_names_out(cat_cols)
X_train_ohe=pd.DataFrame(X_train_ohe, columns=col_names, index=X_train.index)
X_train_ready=pd.concat([X_train_ohe, X_train_scaled], axis=1)


In [270]:
X_test_ohe=ohe.transform(X_test[cat_cols])
X_test_ohe=pd.DataFrame(X_test_ohe, columns=col_names, index=X_test.index)
X_test_ready=pd.concat([X_test_ohe, X_test_scaled], axis=1)

c:\Users\zhiga\anaconda3\envs\mlops26\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [271]:
model=LinearRegression(fit_intercept=True, n_jobs=-1)
model.fit(X_train_ready, y_train)

,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",-1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](91,)","[227.44, 15.31,332.77,..., 62.11,-10.79, -4.41]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](91,)","['neighbourhood_cleansed_Barnet','neighbourhood_cleansed_Bexley', 'neighbourhood_cleansed_Brent',...,'bedrooms','beds','minimum_nights']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,15.66
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,91
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,90


In [272]:
y_pred=model.predict(X_test_ready)

In [273]:
RMSE=root_mean_squared_error(y_test, y_pred)
print('RMSE: ', np.round(RMSE, 4))

RMSE:  267.9416
